In [1]:
#---------------------------------------------------------
# 1. LIBRARIES
#---------------------------------------------------------

import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
#---------------------------------------------------------
# 2. DATA PREPROCESSING
#---------------------------------------------------------

import sys
import os

main_dir = os.path.abspath('..')

sys.path.append(main_dir + '/utils')

from preprocessing import preprocess

train_data = pd.read_csv('../data/adult_train_cleaned.csv')

x_train = train_data.drop('income', axis=1)
y_train = train_data['income']

preprocessor = preprocess(x_train)

In [3]:
#---------------------------------------------------------
# 3. HYPERPARAMETER TUNING & CROSS-VALIDATION
#---------------------------------------------------------

x_train_preprocessed = preprocessor.fit_transform(x_train)
y_train_preprocessed = LabelEncoder().fit_transform(y_train)

model = RandomForestClassifier(
    criterion = 'gini',
    max_depth = None,
    min_impurity_decrease = 0.0,
    bootstrap = True,
    oob_score = True,
    class_weight = 'balanced',
    n_jobs = -1,
    random_state = 42
)

param_grid = {
    'n_estimators': [100, 150],
    'min_samples_split': [2, 5, 8],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 0.2, 0.4]
}

grid = GridSearchCV(
    estimator = model,
    param_grid = param_grid,
    cv = 5,
    scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
    refit = 'f1',
    n_jobs = -1,
    verbose = 2
)

grid.fit(x_train_preprocessed, y_train_preprocessed)

print(grid.best_params_)
print(grid.best_score_)

Fitting 5 folds for each of 72 candidates, totalling 360 fits
{'max_features': 0.2, 'min_samples_leaf': 2, 'min_samples_split': 8, 'n_estimators': 100}
0.710379454089663


In [4]:
#---------------------------------------------------------
# 4. MODEL EVALUATION ON TEST SET
#---------------------------------------------------------

best_model = grid.best_estimator_

model_pipeline = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('classifier', best_model)
], memory = None)

model_pipeline.fit(x_train, y_train)

test_data = pd.read_csv('../data/adult_test_cleaned.csv')

x_test = test_data.drop('income', axis=1)
y_test = test_data['income']

y_pred = model_pipeline.predict(x_test)

print("Classification Report on Test Set:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix on Test Set:")
print(confusion_matrix(y_test, y_pred))

Classification Report on Test Set:
              precision    recall  f1-score   support

       <=50K       0.93      0.86      0.89     12430
        >50K       0.63      0.78      0.70      3846

    accuracy                           0.84     16276
   macro avg       0.78      0.82      0.79     16276
weighted avg       0.86      0.84      0.85     16276

Confusion Matrix on Test Set:
[[10684  1746]
 [  850  2996]]


In [5]:
#---------------------------------------------------------
# 5. MODEL PERSISTENCE
#---------------------------------------------------------

joblib.dump(model_pipeline, '../models/rf_model.joblib')

['../models/rf_model.joblib']